[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/filippolonghi/medleydb-mert-instrument-classification/blob/main/notebooks/08_genre_instrumentation_cooccurrence.ipynb)


# 08 - Genre, Instrumentation, and Co-Occurrence

## Qualitative purpose

This notebook turns completed metadata artifacts into report figures about dataset bias and musical context. It asks which instruments dominate the selected data, which instruments commonly appear together in mixture protocols, and whether genre metadata can explain some model confusions.

## Technical purpose

The notebook reads an existing isolated subset CSV or polyphonic mixture manifest. It does not train models, extract MERT embeddings, write caches, change experiment IDs, or touch the registry. It saves report tables under `results/tables/` and figures under `results/figures/`.

## Why this matters for the report

Instrument frequency and co-occurrence are possible confounders: a model may perform well on common instruments, confuse instruments that often appear together, or inherit genre-specific bias from the dataset. These plots help explain per-class F1, confusion matrices, and polyphonic overlap results without adding another training protocol.


## What you can change

Set `ANALYSIS_SOURCE` to one of `auto`, `isolated_subset`, `synthetic_mixture`, `same_song_mixture`, or `full_mix_manifest`. The notebook continues without genre plots if genre metadata is absent or too sparse.


In [ ]:
from pathlib import Path
import os
import re
import shutil
import sys

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/content/medleydb-mert-instrument-classification"))
RUN_ROOT = Path(os.environ.get("RUN_ROOT", "/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1"))
MEDLEYDB_ROOT = Path(os.environ.get("MEDLEYDB_ROOT", "/content/drive/MyDrive/medleydb_mert_project/MedleyDB"))
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)
if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = RUN_ROOT / "results"
FIGURE_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"
for path in [FIGURE_DIR, TABLE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Run root:", RUN_ROOT)
print("MedleyDB root:", MEDLEYDB_ROOT)


In [ ]:
ANALYSIS_SOURCE = "auto"  # auto | isolated_subset | synthetic_mixture | same_song_mixture | full_mix_manifest
TOP_K_INSTRUMENTS = 20
TOP_K_PAIRS = 30
GENRE_MIN_ROWS = 5

ISOLATED_SUBSET_CSV = RUN_ROOT / "data" / "metadata" / "subset_largest_balanced_medleydb_instrument.csv"
SYNTHETIC_MIXTURE_MANIFEST = RUN_ROOT / "data" / "metadata" / "mixtures" / "largest_balanced_synthetic_k.csv"
SAME_SONG_MIXTURE_MANIFEST = RUN_ROOT / "data" / "metadata" / "mixtures" / "largest_balanced_same_song_same_time.csv"
FULL_MIX_MANIFEST = RUN_ROOT / "data" / "metadata" / "mixtures" / "largest_balanced_full_mix.csv"

SOURCE_PATHS = {
    "isolated_subset": ISOLATED_SUBSET_CSV,
    "synthetic_mixture": SYNTHETIC_MIXTURE_MANIFEST,
    "same_song_mixture": SAME_SONG_MIXTURE_MANIFEST,
    "full_mix_manifest": FULL_MIX_MANIFEST,
}
print("Available analysis sources:")
for name, path in SOURCE_PATHS.items():
    print(f"  {name}: {path} {'[found]' if path.exists() else '[missing]'}")


In [ ]:
def choose_source():
    if ANALYSIS_SOURCE != "auto":
        path = SOURCE_PATHS[ANALYSIS_SOURCE]
        if not path.exists():
            raise FileNotFoundError(f"Selected analysis source is missing: {path}")
        return ANALYSIS_SOURCE, path
    for name in ["full_mix_manifest", "same_song_mixture", "isolated_subset", "synthetic_mixture"]:
        path = SOURCE_PATHS[name]
        if path.exists():
            return name, path
    raise FileNotFoundError("No analysis source found. Build the isolated subset or a mixture manifest first.")


def split_labels(value):
    if pd.isna(value):
        return []
    return [label for label in str(value).split("|") if label]


def prepare_analysis_frame(path, source_name):
    frame = pd.read_csv(path)
    frame = frame.copy()
    if "active_labels" in frame.columns:
        frame["analysis_labels"] = frame["active_labels"].apply(split_labels)
    elif "label_name" in frame.columns:
        frame["analysis_labels"] = frame["label_name"].fillna("").map(lambda value: [str(value)] if str(value) else [])
    elif "medleydb_instrument_label" in frame.columns:
        frame["analysis_labels"] = frame["medleydb_instrument_label"].fillna("").map(lambda value: [str(value)] if str(value) else [])
    elif "coarse_label" in frame.columns:
        frame["analysis_labels"] = frame["coarse_label"].fillna("").map(lambda value: [str(value)] if str(value) else [])
    else:
        raise ValueError("Source must contain active_labels, label_name, medleydb_instrument_label, or coarse_label")
    frame["source_name"] = source_name
    return frame

source_name, source_path = choose_source()
analysis = prepare_analysis_frame(source_path, source_name)
print(f"Using {source_name}: {source_path}")
print(f"Rows: {len(analysis)}")
display(analysis.head())


In [ ]:
def instrument_counts(frame):
    rows = []
    for labels in frame["analysis_labels"]:
        for label in labels:
            rows.append(label)
    counts = pd.Series(rows, dtype="string").value_counts().rename_axis("instrument").reset_index(name="count")
    counts["fraction"] = counts["count"] / max(1, len(frame))
    return counts


def cooccurrence_matrix(frame):
    labels = sorted({label for labels in frame["analysis_labels"] for label in labels})
    matrix = pd.DataFrame(0, index=labels, columns=labels, dtype=int)
    for labels in frame["analysis_labels"]:
        active = sorted(set(labels))
        for a in active:
            for b in active:
                matrix.loc[a, b] += 1
    return matrix


def normalize_cooccurrence(raw):
    supports = pd.Series(np.diag(raw), index=raw.index).replace(0, np.nan)
    normalized = raw.div(supports, axis=0).fillna(0.0)
    return normalized


def top_pairs(raw, k=30):
    rows = []
    labels = list(raw.index)
    for i, a in enumerate(labels):
        for b in labels[i + 1:]:
            count = int(raw.loc[a, b])
            if count:
                support_a = int(raw.loc[a, a])
                support_b = int(raw.loc[b, b])
                rows.append({
                    "instrument_a": a,
                    "instrument_b": b,
                    "cooccurrence_count": count,
                    "fraction_of_a": count / support_a if support_a else 0.0,
                    "fraction_of_b": count / support_b if support_b else 0.0,
                })
    return pd.DataFrame(rows).sort_values("cooccurrence_count", ascending=False).head(k) if rows else pd.DataFrame(columns=["instrument_a", "instrument_b", "cooccurrence_count", "fraction_of_a", "fraction_of_b"])

counts = instrument_counts(analysis)
raw_matrix = cooccurrence_matrix(analysis)
normalized_matrix = normalize_cooccurrence(raw_matrix)
pairs = top_pairs(raw_matrix, TOP_K_PAIRS)

prefix = f"{source_name}_instrumentation"
counts.to_csv(TABLE_DIR / f"{prefix}_instrument_counts.csv", index=False)
raw_matrix.to_csv(TABLE_DIR / f"{prefix}_cooccurrence_raw.csv", index_label="instrument")
normalized_matrix.to_csv(TABLE_DIR / f"{prefix}_cooccurrence_normalized.csv", index_label="instrument")
pairs.to_csv(TABLE_DIR / f"{prefix}_top_{TOP_K_PAIRS}_cooccurring_pairs.csv", index=False)

print("Saved tables to", TABLE_DIR)
display(counts.head(TOP_K_INSTRUMENTS))
display(pairs.head(10))


In [ ]:
def save_bar_plot(counts, path):
    top = counts.head(TOP_K_INSTRUMENTS).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(top))))
    ax.barh(top["instrument"], top["count"], color="#4C78A8")
    ax.set_xlabel("Segments or mixtures")
    ax.set_ylabel("Instrument")
    ax.set_title(f"Top instruments: {source_name}")
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)


def save_heatmap(matrix, path):
    if matrix.empty:
        print("No co-occurrence heatmap saved because the matrix is empty.")
        return
    order = raw_matrix.sum(axis=1).sort_values(ascending=False).head(TOP_K_INSTRUMENTS).index
    plot_matrix = matrix.loc[order, order]
    fig_size = max(6, 0.45 * len(order))
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    im = ax.imshow(plot_matrix.values, cmap="Blues", vmin=0)
    ax.set_xticks(range(len(order)))
    ax.set_yticks(range(len(order)))
    ax.set_xticklabels(order, rotation=45, ha="right")
    ax.set_yticklabels(order)
    ax.set_title(f"Normalized co-occurrence: {source_name}")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)

bar_path = FIGURE_DIR / f"{prefix}_instrument_frequency.png"
heatmap_path = FIGURE_DIR / f"{prefix}_cooccurrence_heatmap.png"
save_bar_plot(counts, bar_path)
save_heatmap(normalized_matrix, heatmap_path)
print("Saved figures:", bar_path, heatmap_path)


In [ ]:
genre_columns = [column for column in ["genre", "genres", "style"] if column in analysis.columns]
genre_warning = None
if not genre_columns:
    genre_warning = "Genre metadata is missing; genre x instrument outputs were skipped."
else:
    genre_column = genre_columns[0]
    genre_frame = analysis[[genre_column, "analysis_labels"]].copy()
    genre_frame[genre_column] = genre_frame[genre_column].fillna("").astype(str).str.strip()
    genre_frame = genre_frame[genre_frame[genre_column] != ""]
    if len(genre_frame) < GENRE_MIN_ROWS:
        genre_warning = f"Genre metadata is sparse ({len(genre_frame)} usable rows); genre x instrument outputs were skipped."
    else:
        rows = []
        for _, row in genre_frame.iterrows():
            for label in row["analysis_labels"]:
                rows.append({"genre": row[genre_column], "instrument": label})
        genre_counts = pd.DataFrame(rows).value_counts(["genre", "instrument"]).reset_index(name="count")
        genre_matrix = genre_counts.pivot(index="genre", columns="instrument", values="count").fillna(0).astype(int)
        genre_table_path = TABLE_DIR / f"{prefix}_genre_by_instrument.csv"
        genre_plot_path = FIGURE_DIR / f"{prefix}_genre_by_instrument.png"
        genre_matrix.to_csv(genre_table_path, index_label="genre")
        top_instruments = counts.head(min(TOP_K_INSTRUMENTS, len(counts)))["instrument"].tolist()
        plot_matrix = genre_matrix.reindex(columns=top_instruments, fill_value=0)
        fig, ax = plt.subplots(figsize=(max(7, 0.4 * len(top_instruments)), max(4, 0.35 * len(plot_matrix))))
        im = ax.imshow(plot_matrix.values, cmap="Greens", aspect="auto")
        ax.set_xticks(range(len(plot_matrix.columns)))
        ax.set_xticklabels(plot_matrix.columns, rotation=45, ha="right")
        ax.set_yticks(range(len(plot_matrix.index)))
        ax.set_yticklabels(plot_matrix.index)
        ax.set_title(f"Genre x instrument: {source_name}")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.tight_layout()
        fig.savefig(genre_plot_path, dpi=180, bbox_inches="tight")
        plt.close(fig)
        print("Saved genre outputs:", genre_table_path, genre_plot_path)

if genre_warning:
    print("Warning:", genre_warning)


In [ ]:
print("Automatic interpretation")
if counts.empty:
    print("- No instrument labels were found in the selected source.")
else:
    top_labels = counts.head(5).apply(lambda row: f"{row['instrument']} ({int(row['count'])})", axis=1).tolist()
    print("- Most common instruments:", ", ".join(top_labels))
if pairs.empty:
    print("- No off-diagonal co-occurring pairs were found. This is expected for single-label isolated subsets.")
else:
    pair_text = pairs.head(5).apply(lambda row: f"{row['instrument_a']} + {row['instrument_b']} ({int(row['cooccurrence_count'])})", axis=1).tolist()
    print("- Most common co-occurring pairs:", ", ".join(pair_text))
if 'genre_warning' in globals() and genre_warning:
    print("-", genre_warning)
else:
    print("- Genre metadata was sufficient for a genre x instrument table/plot.")
print("- Use these outputs to contextualize class imbalance, frequent instrument combinations, and plausible confusion pairs in the final report.")
